# Практическая работа №2. Визуализация данных в Python

В работе исследуется дневной набор Bike Sharing за 2011-2012 годы.
Для многомерной визуализации используется встроенный набор рукописных
цифр `sklearn.datasets.load_digits` (1797 объектов, 64 признака).

**Студент: Албахтин И.В., ИНБО-12-23.**

In [1]:
from __future__ import annotations

import io
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sklearn
import umap
from matplotlib.lines import Line2D
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE, trustworthiness
from sklearn.preprocessing import StandardScaler


PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "practice_1" / "FiresRu.csv").is_file())
ROOT = PROJECT / "practice_2"

DATA_PATH = ROOT / "data" / "day.csv"
OUTPUT = PROJECT / "АлбахтинИВ" / "Результаты" / "Визуализация"
FIGURES = OUTPUT / "figures"
INTERACTIVE = OUTPUT / "interactive"
FIGURES.mkdir(parents=True, exist_ok=True)
INTERACTIVE.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "figure.dpi": 120,
    }
)

/home/albakhtin/tiabd/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Загрузка и описание данных

`day.csv` содержит ежедневные показатели системы Capital Bikeshare в
Вашингтоне: дату, сезон, календарные признаки, погоду, число случайных и
зарегистрированных пользователей и общее число поездок.

In [2]:
df = pd.read_csv(DATA_PATH, parse_dates=["dteday"])

print("Размер исходной таблицы:", df.shape)
print("\nПервые пять строк:")
print(df.head().to_string(index=False))

buffer = io.StringIO()
df.info(buf=buffer, memory_usage="deep")
print("\nРезультат info():")
print(buffer.getvalue())

missing = df.isna().sum()
duplicates = int(df.duplicated().sum())
print("Пропуски по столбцам:")
print(missing.to_string())
print("Полные дубликаты:", duplicates)

Размер исходной таблицы: (731, 16)

Первые пять строк:
 instant     dteday  season  yr  mnth  holiday  weekday  workingday  weathersit     temp    atemp      hum  windspeed  casual  registered  cnt
       1 2011-01-01       1   0     1        0        6           0           2 0.344167 0.363625 0.805833   0.160446     331         654  985
       2 2011-01-02       1   0     1        0        0           0           2 0.363478 0.353739 0.696087   0.248539     131         670  801
       3 2011-01-03       1   0     1        0        1           1           1 0.196364 0.189405 0.437273   0.248309     120        1229 1349
       4 2011-01-04       1   0     1        0        2           1           1 0.200000 0.212122 0.590435   0.160296     108        1454 1562
       5 2011-01-05       1   0     1        0        3           1           1 0.226957 0.229270 0.436957   0.186900      82        1518 1600

Результат info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 

Пропусков и полных дубликатов нет. Температура и ветер сохраняются в нормированной шкале. Влажность переводится в проценты по метаданным. Сезоны подписываются исходными кодами, поскольку описания набора дают разные словесные расшифровки.

In [3]:
season_names = {1: "Код сезона 1", 2: "Код сезона 2", 3: "Код сезона 3", 4: "Код сезона 4"}
weather_names = {
    1: "Ясно или малооблачно",
    2: "Туман или облачно",
    3: "Небольшие осадки",
    4: "Сильные осадки",
}

clean = df.copy()
clean["season_name"] = clean["season"].map(season_names).astype("category")
clean["weather_name"] = clean["weathersit"].map(weather_names).astype("category")
clean["humidity_pct"] = clean["hum"] * 100

assert int(clean.isna().sum().sum()) == 0
assert (clean["casual"] + clean["registered"] == clean["cnt"]).all()

clean.to_csv(OUTPUT / "bike_sharing_visualization_clean.csv", index=False)
clean.to_parquet(OUTPUT / "bike_sharing_visualization_clean.parquet", index=False)

data_summary = {
    "rows": int(len(clean)),
    "columns_source": int(df.shape[1]),
    "date_min": clean["dteday"].min().date().isoformat(),
    "date_max": clean["dteday"].max().date().isoformat(),
    "missing_values": int(clean.isna().sum().sum()),
    "duplicate_rows": duplicates,
    "total_rentals": int(clean["cnt"].sum()),
    "mean_daily_rentals": round(float(clean["cnt"].mean()), 2),
    "median_daily_rentals": round(float(clean["cnt"].median()), 2),
    "max_daily_rentals": int(clean["cnt"].max()),
}
print("Сводка:", json.dumps(data_summary, ensure_ascii=False, indent=2))

Сводка: {
  "rows": 731,
  "columns_source": 16,
  "date_min": "2011-01-01",
  "date_max": "2012-12-31",
  "missing_values": 0,
  "duplicate_rows": 0,
  "total_rentals": 3292679,
  "mean_daily_rentals": 4504.35,
  "median_daily_rentals": 4548.0,
  "max_daily_rentals": 8714
}


## 2. Столбчатая диаграмма Plotly

Чтобы подписи дат оставались читаемыми, дневные поездки агрегируются по
месяцам. Цвет каждого столбца определяется тем же количественным показателем.

In [4]:
monthly = (
    clean.set_index("dteday")
    .resample("MS")[["casual", "registered", "cnt"]]
    .sum()
    .reset_index()
)
monthly["month_label"] = monthly["dteday"].dt.strftime("%Y-%m")

bar = go.Figure(
    data=[
        go.Bar(
            x=monthly["month_label"],
            y=monthly["cnt"],
            name="Поездки",
            marker=dict(
                color=monthly["cnt"],
                coloraxis="coloraxis",
                line=dict(color="black", width=2),
            ),
            hovertemplate="Месяц: %{x}<br>Поездки: %{y:,}<extra></extra>",
        )
    ]
)
bar.update_layout(
    title=dict(text="Количество поездок по месяцам", x=0.5, font=dict(size=20)),
    xaxis=dict(
        title=dict(text="Месяц", font=dict(size=16)),
        tickfont=dict(size=14),
        tickangle=315,
        gridwidth=2,
        gridcolor="ivory",
    ),
    yaxis=dict(
        title=dict(text="Количество поездок", font=dict(size=16)),
        tickfont=dict(size=14),
        gridwidth=2,
        gridcolor="ivory",
        rangemode="tozero",
    ),
    coloraxis=dict(colorscale="Viridis", colorbar=dict(title="Поездки")),
    template="plotly_white",
    autosize=True,
    height=700,
    margin=dict(l=10, r=10, t=65, b=10),
    showlegend=False,
)
bar.write_html(INTERACTIVE / "01_monthly_bar.html", include_plotlyjs="cdn")
bar.write_image(FIGURES / "01_monthly_bar.png", width=1400, height=700, scale=1.5)
bar.show()

Максимум месячного спроса приходится на сентябрь 2012 года. В 2012 году
почти все месяцы заметно выше соответствующих месяцев 2011 года.

## 3. Круговая диаграмма Plotly

In [5]:
season_totals = (
    clean.groupby("season_name", observed=True)["cnt"]
    .sum()
    .reindex(list(season_names.values()))
)

normalized = (season_totals - season_totals.min()) / (
    season_totals.max() - season_totals.min()
)
pie_colors = [
    f"rgb({int(68 + 185 * v)}, {int(1 + 230 * v)}, {int(84 + 60 * (1-v))})"
    for v in normalized
]

pie = go.Figure(
    data=[
        go.Pie(
            labels=season_totals.index,
            values=season_totals.values,
            textinfo="label+percent",
            textfont=dict(size=14),
            marker=dict(colors=pie_colors, line=dict(color="black", width=2)),
            sort=False,
            hovertemplate="%{label}: %{value:,} поездок (%{percent})<extra></extra>",
        )
    ]
)
pie.update_layout(
    title=dict(text="Распределение поездок по сезонам", x=0.5, font=dict(size=20)),
    template="plotly_white",
    autosize=True,
    height=700,
    margin=dict(l=10, r=10, t=65, b=10),
    showlegend=False,
)
pie.write_html(INTERACTIVE / "02_season_pie.html", include_plotlyjs="cdn")
pie.write_image(FIGURES / "02_season_pie.png", width=1100, height=700, scale=1.5)
pie.show()

Наибольшая доля поездок приходится на осень, наименьшая — на весну. Разрыв
между сезонами заметен, но ни одна категория не является исчезающе малой.

## 4. Линейные графики Matplotlib

Временной параметр — месяц; зависимые показатели — число поездок случайных,
зарегистрированных и всех пользователей.

In [6]:
series = [
    ("casual", "Случайные пользователи"),
    ("registered", "Зарегистрированные пользователи"),
    ("cnt", "Все поездки"),
]

fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
for ax, (column, title) in zip(axes, series, strict=True):
    ax.plot(
        monthly["dteday"],
        monthly[column],
        color="crimson",
        linewidth=2,
        marker="o",
        markersize=6,
        markerfacecolor="white",
        markeredgecolor="black",
        markeredgewidth=2,
    )
    ax.set_title(title)
    ax.set_ylabel("Количество")
    ax.grid(linewidth=2, color="mistyrose")
axes[-1].set_xlabel("Месяц")
fig.suptitle("Динамика спроса по типам пользователей", fontsize=16, y=0.995)
fig.autofmt_xdate(rotation=45)
fig.tight_layout()
fig.savefig(FIGURES / "03_monthly_lines.png", bbox_inches="tight")
plt.show()

monthly_correlations = monthly[["casual", "registered", "cnt"]].corr().round(3)
print("Корреляции месячных рядов:")
print(monthly_correlations.to_string())

Корреляции месячных рядов:
            casual  registered    cnt
casual       1.000       0.864  0.922
registered   0.864       1.000  0.991
cnt          0.922       0.991  1.000


Все три ряда имеют общую сезонную форму и растут во втором году. Основной
вклад в общее число поездок вносит зарегистрированная аудитория; корреляция
месячных рядов `registered` и `cnt` выше, чем у `casual` и `cnt`.

## 5. t-SNE для набора рукописных цифр

In [7]:
digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target
X_scaled = StandardScaler().fit_transform(X)

print("Размер набора digits:", X.shape)
print("Классы:", np.unique(y))
print("Пропуски:", int(np.isnan(X_scaled).sum()))

perplexities = [5, 30, 50]
tsne_results: dict[int, np.ndarray] = {}
timings: list[dict[str, float | int | str]] = []

for perplexity in perplexities:
    started = time.perf_counter()
    model = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        max_iter=1000,
        random_state=RANDOM_STATE,
    )
    embedding = model.fit_transform(X_scaled)
    elapsed = time.perf_counter() - started
    quality = trustworthiness(X_scaled, embedding, n_neighbors=10)
    tsne_results[perplexity] = embedding
    timings.append(
        {
            "algorithm": "t-SNE",
            "parameters": f"perplexity={perplexity}",
            "seconds": round(elapsed, 4),
            "trustworthiness_k10": round(float(quality), 4),
        }
    )
    print(
        f"t-SNE perplexity={perplexity}: {elapsed:.3f} s, "
        f"trustworthiness@10={quality:.4f}, KL={model.kl_divergence_:.4f}"
    )

Размер набора digits: (1797, 64)
Классы: [0 1 2 3 4 5 6 7 8 9]
Пропуски: 0


t-SNE perplexity=5: 2.602 s, trustworthiness@10=0.9817, KL=0.9799


t-SNE perplexity=30: 2.735 s, trustworthiness@10=0.9861, KL=0.8374


t-SNE perplexity=50: 8.706 s, trustworthiness@10=0.9857, KL=0.7741


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), constrained_layout=True)
for ax, perplexity in zip(axes, perplexities, strict=True):
    embedding = tsne_results[perplexity]
    scatter = ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=y,
        cmap="tab10",
        s=10,
        alpha=0.78,
        linewidths=0,
    )
    ax.set_title(f"Перплексия {perplexity}")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("t-SNE: влияние перплексии", fontsize=16)
handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor=plt.cm.tab10(i),
           markersize=7, label=str(i))
    for i in range(10)
]
fig.legend(handles=handles, title="Цифра", loc="center right", bbox_to_anchor=(1.045, 0.5))
fig.savefig(FIGURES / "04_tsne_perplexities.png", bbox_inches="tight")
plt.show()

При перплексии 5 локальные подгруппы сильнее дробятся. Значения 30 и 50
формируют более цельные кластеры; некоторые классы всё равно частично
пересекаются из-за сходства начертания цифр.

## 6. UMAP с разными параметрами

In [9]:
umap_configs = [(5, 0.0), (15, 0.1), (50, 0.5)]
umap_results: dict[tuple[int, float], np.ndarray] = {}

for n_neighbors, min_dist in umap_configs:
    started = time.perf_counter()
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="euclidean",
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    embedding = reducer.fit_transform(X_scaled)
    elapsed = time.perf_counter() - started
    quality = trustworthiness(X_scaled, embedding, n_neighbors=10)
    umap_results[(n_neighbors, min_dist)] = embedding
    timings.append(
        {
            "algorithm": "UMAP",
            "parameters": f"n_neighbors={n_neighbors}, min_dist={min_dist}",
            "seconds": round(elapsed, 4),
            "trustworthiness_k10": round(float(quality), 4),
        }
    )
    print(
        f"UMAP n_neighbors={n_neighbors}, min_dist={min_dist}: "
        f"{elapsed:.3f} s, trustworthiness@10={quality:.4f}"
    )

UMAP n_neighbors=5, min_dist=0.0: 6.268 s, trustworthiness@10=0.9768


UMAP n_neighbors=15, min_dist=0.1: 2.264 s, trustworthiness@10=0.9717


UMAP n_neighbors=50, min_dist=0.5: 4.752 s, trustworthiness@10=0.9468


In [10]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), constrained_layout=True)
for ax, config in zip(axes, umap_configs, strict=True):
    embedding = umap_results[config]
    ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=y,
        cmap="tab10",
        s=10,
        alpha=0.78,
        linewidths=0,
    )
    ax.set_title(f"n_neighbors={config[0]}, min_dist={config[1]}")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("UMAP: влияние n_neighbors и min_dist", fontsize=16)
fig.legend(handles=handles, title="Цифра", loc="center right", bbox_to_anchor=(1.045, 0.5))
fig.savefig(FIGURES / "05_umap_parameters.png", bbox_inches="tight")
plt.show()

Малое число соседей и `min_dist=0` дают компактные локальные группы. При
увеличении обоих параметров точки распределяются свободнее, а глобальная
взаимная ориентация классов становится заметнее.

## 7. Сравнение времени и качества локальной структуры

In [11]:
timings_df = pd.DataFrame(timings)
timings_df.to_csv(OUTPUT / "dimension_reduction_timings.csv", index=False)
print(timings_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(11.5, 5.5))
labels = [
    f"{row.algorithm}\n{row.parameters.replace(', ', chr(10))}"
    for row in timings_df.itertuples()
]
colors = ["#c51b7d" if name == "t-SNE" else "#4d9221" for name in timings_df["algorithm"]]
bars = ax.bar(labels, timings_df["seconds"], color=colors, edgecolor="black", linewidth=1.2)
ax.bar_label(bars, fmt="%.2f с", padding=3)
ax.set_ylabel("Время, секунд")
ax.set_title("Измеренное время снижения размерности")
ax.grid(axis="y", color="#e6e6e6", linewidth=1.2)
ax.set_axisbelow(True)
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
fig.tight_layout()
fig.savefig(FIGURES / "06_runtime_comparison.png", bbox_inches="tight")
plt.show()

tsne_mean = float(timings_df.loc[timings_df["algorithm"] == "t-SNE", "seconds"].mean())
umap_mean = float(timings_df.loc[timings_df["algorithm"] == "UMAP", "seconds"].mean())
runtime_summary = {
    "python": __import__("sys").version.split()[0],
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "matplotlib": __import__("matplotlib").__version__,
    "plotly": __import__("plotly").__version__,
    "scikit_learn": sklearn.__version__,
    "umap_learn": umap.__version__,
    "random_state": RANDOM_STATE,
    "tsne_mean_seconds": round(tsne_mean, 4),
    "umap_mean_seconds": round(umap_mean, 4),
    "umap_to_tsne_ratio": round(umap_mean / tsne_mean, 4),
    "note": "Первый запуск UMAP включает JIT-компиляцию numba и поэтому обычно медленнее последующих.",
}

summary = {
    "bike_sharing": data_summary,
    "season_totals": {str(k): int(v) for k, v in season_totals.items()},
    "monthly_peak": {
        "month": monthly.loc[monthly["cnt"].idxmax(), "month_label"],
        "rentals": int(monthly["cnt"].max()),
    },
    "monthly_correlations": monthly_correlations.to_dict(),
    "dimension_reduction": runtime_summary,
    "timings": timings,
}
(OUTPUT / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(json.dumps(runtime_summary, ensure_ascii=False, indent=2))

algorithm                   parameters  seconds  trustworthiness_k10
    t-SNE                 perplexity=5   2.6018               0.9817
    t-SNE                perplexity=30   2.7353               0.9861
    t-SNE                perplexity=50   8.7063               0.9857
     UMAP  n_neighbors=5, min_dist=0.0   6.2682               0.9768
     UMAP n_neighbors=15, min_dist=0.1   2.2638               0.9717
     UMAP n_neighbors=50, min_dist=0.5   4.7525               0.9468


{
  "python": "3.10.12",
  "pandas": "2.2.3",
  "numpy": "2.2.6",
  "matplotlib": "3.10.8",
  "plotly": "6.3.1",
  "scikit_learn": "1.7.2",
  "umap_learn": "0.5.9.post2",
  "random_state": 42,
  "tsne_mean_seconds": 4.6811,
  "umap_mean_seconds": 4.4282,
  "umap_to_tsne_ratio": 0.946,
  "note": "Первый запуск UMAP включает JIT-компиляцию numba и поэтому обычно медленнее последующих."
}


## 8. Итоговые выводы

- Спрос имеет выраженную сезонность и существенно вырос во втором году.
- Зарегистрированные пользователи формируют основную часть месячного спроса.
- Plotly удобен для интерактивной проверки точных значений; Matplotlib даёт
  полный контроль над статическим многопанельным рисунком.
- t-SNE чувствителен к перплексии и хорошо показывает локальные кластеры, но
  расстояния между удалёнными кластерами нельзя трактовать буквально.
- UMAP позволяет управлять балансом локальной и глобальной структуры через
  `n_neighbors` и `min_dist`; первый запуск включает JIT-компиляцию.

# Практическая работа №2. Подробные ответы на контрольные вопросы

1. **Из каких основных объектов состоит график Matplotlib?**

   **Простыми словами.** График Matplotlib устроен как набор вложенных объектов. `Figure` — весь лист или холст. `Axes` — отдельная область, где строятся данные. `Axis` — конкретная координатная ось X или Y с диапазоном, делениями и подписями. Линии, точки, текст, легенда и другие видимые элементы называются `Artist`.

   **Как это работает.** Команда `fig, ax = plt.subplots()` создаёт `Figure` и один `Axes`. Затем `ax.plot(...)` добавляет линию именно в эту область. У `ax` есть две оси: `ax.xaxis` и `ax.yaxis`. Если написать `plt.subplots(2, 2)`, одна Figure будет содержать четыре Axes, то есть четыре отдельных подграфика.

   **Пример и вывод.** В практической работе рисунок с тремя временными рядами состоит из одной Figure и трёх Axes. Заголовок всего рисунка относится к Figure, а название каждого подграфика — к соответствующему Axes. Такое разделение важно для понимания объектного интерфейса: мы изменяем конкретную часть рисунка, а не «график вообще».

2. **Чем интерфейс pyplot отличается от объектного интерфейса Matplotlib?**

   **Простыми словами.** `pyplot` — набор команд, похожий на работу с текущим активным рисунком: `plt.plot()`, `plt.title()`, `plt.xlabel()`. Объектный интерфейс сначала создаёт объекты `fig` и `ax`, а затем явно вызывает их методы: `ax.plot()`, `ax.set_title()`, `ax.set_xlabel()`.

   **Как это работает.** `pyplot` хранит внутреннее состояние и решает, какой рисунок сейчас считается текущим. Для одного простого графика это удобно и коротко. При нескольких подграфиках легко случайно изменить не ту область. Объектный интерфейс не полагается на скрытое состояние: переменная `ax` однозначно указывает, куда добавляется линия, сетка или подпись.

   **Пример и вывод.** В цикле можно написать `for ax, column in zip(axes, columns): ax.plot(...)` и настроить каждую панель отдельно. Поэтому для отчётов, сложных композиций и повторно используемых функций предпочтителен объектный интерфейс. `pyplot` всё равно применяется для создания Figure и быстрого показа, но основная настройка выполняется через объекты.

3. **Зачем явно вызывать figure() или subplots()?**

   **Простыми словами.** Эти функции создают новый рисунок и позволяют заранее определить его размер и устройство. Без явного создания Matplotlib может использовать уже активный рисунок, из-за чего новый график случайно наложится на старый или получит неподходящий размер.

   **Как это работает.** `plt.figure(figsize=(8, 5))` создаёт Figure размером 8 на 5 дюймов, после чего pyplot работает с ней как с текущей. `fig, ax = plt.subplots()` одновременно создаёт Figure и Axes. Параметры `nrows` и `ncols` задают сетку подграфиков, а `figsize` — физический размер всего рисунка.

   **Пример и вывод.** Для трёх рядов Bike Sharing используется `plt.subplots(3, 1, ...)`: три области располагаются вертикально и делят общую временную ось. Явное создание делает код воспроизводимым: размер, число панелей и объект, который изменяется, видны непосредственно в программе.

4. **Для чего используется zorder?**

   **Простыми словами.** `zorder` задаёт, какой объект рисуется поверх другого. Это похоже на слои в графическом редакторе: объект с большим значением находится ближе к зрителю, с меньшим — позади.

   **Как это работает.** Если линия, точки и сетка занимают одно место, порядок их отрисовки влияет на читаемость. Например, сетке можно задать `zorder=0`, столбцам `zorder=2`, а подписям `zorder=3`. Значение не меняет координаты данных, толщину или цвет; оно влияет только на перекрытие элементов.

   **Пример и вывод.** На scatter plot крупные точки могут закрывать линию тренда. Если линии задать больший `zorder`, она останется видимой. Параметр особенно полезен для фоновой сетки, выделенных наблюдений, доверительных интервалов и аннотаций. Следует задавать его осмысленно, иначе важные элементы могут оказаться скрытыми.

5. **В чём различие Plotly Express и graph_objects?**

   **Простыми словами.** Plotly Express — высокоуровневый интерфейс: одна функция быстро строит типовой график из DataFrame. `graph_objects` — низкоуровневый интерфейс, где рисунок собирают из отдельных следов и подробно настраивают макет.

   **Как это работает.** В Express можно написать `px.bar(df, x="month", y="cnt", color="cnt")`, и библиотека сама создаст Figure, следы, легенду и подписи. В `graph_objects` создают `go.Figure()`, добавляют `go.Bar`, `go.Scatter` или другие trace-объекты, а затем вызывают `update_layout()` и `update_xaxes()`. Кода больше, зато доступно больше деталей.

   **Пример и вывод.** В этой работе `go.Bar` используется для явной настройки границ столбцов, цветовой шкалы, высоты и полей рисунка. Express удобен для разведочного анализа и быстрых графиков. `graph_objects` лучше подходит, когда нужны несколько типов следов, вторичная ось, сложные hover-подсказки или точное оформление.

6. **Почему Plotly называют интерактивной библиотекой?**

   **Простыми словами.** Пользователь может взаимодействовать с готовым рисунком, а не только смотреть на статичную картинку. В браузере доступны наведение мыши, увеличение, перемещение области просмотра, включение и выключение рядов и сохранение изображения.

   **Как это работает.** Plotly преобразует описание Figure в JavaScript-график. Данные и настройки могут быть сохранены в HTML вместе с кодом визуализации. Поэтому для просмотра такого файла не требуется запущенный Python: браузер выполняет интерактивную часть самостоятельно. В notebook график отображается через встроенный HTML/JavaScript-рендерер.

   **Пример и вывод.** На столбчатой диаграмме можно навести курсор на месяц и увидеть точное число поездок, а затем приблизить интересующий период. Интерактивность полезна при исследовании данных. Для печатного отчёта всё равно нужна статическая версия, потому что наведение и масштабирование на бумаге недоступны.

7. **Что показывает цветовая шкала в столбчатой диаграмме этой работы?**

   **Простыми словами.** Цвет каждого столбца кодирует то же значение, что и его высота: месячное число поездок. Низкие значения показаны одним краем шкалы, высокие — другим. Справа расположена легенда, связывающая цвет с числом.

   **Как это работает.** В `go.Bar` массив `monthly["cnt"]` передан одновременно в `y` и в `marker.color`. Параметр `coloraxis="coloraxis"` подключает общую непрерывную шкалу. Такое кодирование называют избыточным, потому что один показатель выражается двумя визуальными каналами.

   **Пример и вывод.** Высокий жёлтый столбец сентября 2012 года одновременно выделяется высотой и цветом. Это ускоряет поиск максимумов, но цвет не добавляет новый признак. Если цвет должен был показывать год или сезон, в `marker.color` нужно было бы передать соответствующую категорию и использовать дискретную легенду.

8. **Почему круговую диаграмму не следует перегружать категориями?**

   **Простыми словами.** В круговой диаграмме величины сравниваются по углам и площадям секторов. Человек делает такое сравнение менее точно, чем сравнение длины столбцов. Когда секторов много, они становятся узкими, цвета повторяются, а подписи сталкиваются.

   **Как это работает.** Круг хорошо читается, если категорий мало, их сумма действительно образует целое и различия достаточно заметны. При десяти похожих долях зритель уже не может уверенно определить порядок. Маленькие сектора почти исчезают, а вынесенные подписи занимают больше места, чем сами данные.

   **Пример и вывод.** В практической работе всего четыре сезона, поэтому круговая диаграмма остаётся понятной и показывает долю каждого сезона в общем числе поездок. Если бы категорий было двадцать, лучше использовать отсортированную горизонтальную bar chart. Редкие категории иногда объединяют в «Другие», но только если это не скрывает важный смысл.

9. **Как выбирать между Matplotlib и Plotly?**

   **Простыми словами.** Выбор определяется тем, где и как будет использоваться график. Matplotlib удобен для статических изображений и точного контроля над каждой деталью. Plotly удобен, когда пользователь должен исследовать рисунок в notebook или браузере.

   **Как это работает.** Matplotlib хорошо экспортирует PNG, SVG и PDF, поддерживает сложные научные композиции и обычно предсказуем в печатных документах. Plotly автоматически предоставляет hover-подсказки, масштабирование, панорамирование и управление легендой. За это приходится платить более тяжёлым HTML и зависимостью от браузерного JavaScript.

   **Пример и вывод.** Месячные ряды в отчёте сделаны Matplotlib, потому что важна статическая трёхпанельная композиция. Столбчатая и круговая диаграммы Plotly удобны для просмотра точных значений. Часто используют обе библиотеки: Plotly — на этапе исследования, Matplotlib — для окончательной иллюстрации в отчёте.

10. **Какую задачу решает снижение размерности?**

   **Простыми словами.** Объект может описываться десятками или тысячами признаков, но человек способен непосредственно увидеть только две или три координаты. Снижение размерности преобразует исходные признаки в меньшее число новых координат, стараясь сохранить важные отношения между объектами.

   **Как это работает.** Алгоритм получает матрицу, где строки — объекты, а столбцы — признаки. На выходе он создаёт, например, две координаты для каждой строки. Разные методы сохраняют разную структуру: PCA ориентируется на общую линейную вариативность, t-SNE — на локальные соседства, UMAP — на граф соседей и сочетание локальной и более общей структуры.

   **Пример и вывод.** Каждая рукописная цифра в `load_digits` описана 64 интенсивностями пикселей. t-SNE и UMAP превращают эти 64 признака в две координаты, чтобы точки можно было показать на плоскости. Проекция помогает увидеть группы и ошибки, но не хранит всю исходную информацию и не заменяет количественную проверку.

11. **Что означает перплексия в t-SNE?**

   **Простыми словами.** Перплексия приблизительно задаёт, сколько ближайших соседей алгоритм учитывает при описании локального окружения каждой точки. Это не точное число соседей, но полезная интуиция для выбора масштаба анализа.

   **Как это работает.** t-SNE преобразует расстояния между объектами в вероятности соседства. Для каждой точки подбирается ширина локального распределения так, чтобы его эффективный размер соответствовал перплексии. Малое значение сильнее ориентируется на очень близкие точки и может раздробить данные. Большое значение учитывает более широкое окружение и сглаживает мелкие детали.

   **Пример и вывод.** В работе проверены значения 5, 30 и 50. При 5 группы распадаются на более мелкие фрагменты; 30 и 50 дают более цельные классы. Перплексия должна быть заметно меньше числа объектов, а вывод желательно проверять на нескольких значениях, потому что одна картинка может отражать настройку алгоритма, а не устойчивую структуру данных.

12. **Почему для t-SNE нужно фиксировать random_state?**

   **Простыми словами.** t-SNE использует случайность при начальном размещении точек и в процессе оптимизации. Поэтому два запуска с одинаковыми данными и параметрами могут дать повёрнутые, отражённые или иначе расположенные группы.

   **Как это работает.** `random_state=42` задаёт начальное состояние генератора псевдослучайных чисел. При одинаковых версиях библиотек, данных и параметрах запуск становится повторяемым. Число 42 не обладает специальными свойствами; можно выбрать другое фиксированное целое значение.

   **Пример и вывод.** Если сравниваются перплексии 5, 30 и 50, случайное состояние должно быть одинаковым. Иначе различия между рисунками частично объяснялись бы разной случайной инициализацией. Фиксация `random_state` нужна для честного эксперимента и воспроизводимого отчёта, но не гарантирует совпадение между сильно различающимися версиями библиотек или платформами.

13. **Можно ли трактовать расстояния и размеры кластеров t-SNE буквально?**

   **Простыми словами.** Нет. t-SNE старается расположить близких соседей рядом, но не обязан сохранять точные расстояния между далёкими группами, их площадь, ориентацию или относительное положение на плоскости.

   **Как это работает.** Для размещения всех точек в двух измерениях алгоритм искажает исходную геометрию. Плотность группы на картинке зависит от параметров и оптимизации. Два кластера могут выглядеть далеко друг от друга, хотя в исходном пространстве это расстояние не имеет такого масштаба. Поворот или зеркальное отражение всей проекции вообще не меняет её смысла.

   **Пример и вывод.** На рисунке цифры 3 и 9 могут находиться рядом из-за похожих пиксельных шаблонов, но расстояние в сантиметрах между их облаками нельзя использовать как числовую меру сходства. Надёжнее анализировать локальных соседей, проверять несколько запусков и использовать метрики в исходном 64-мерном пространстве.

14. **За что отвечают n_neighbors и min_dist в UMAP?**

   **Простыми словами.** `n_neighbors` определяет размер окружения, которое UMAP рассматривает вокруг каждой точки. `min_dist` задаёт, насколько близко точки могут располагаться в готовой проекции. Первый параметр управляет масштабом структуры, второй — визуальной плотностью групп.

   **Как это работает.** При малом `n_neighbors` алгоритм сильнее сохраняет очень локальные отношения и может показать много небольших групп. При большом значении учитывается более широкое окружение, поэтому проявляется общая структура. Малый `min_dist` разрешает точкам собираться в плотные кластеры; большой оставляет между соседями больше пространства.

   **Пример и вывод.** В конфигурации `n_neighbors=5, min_dist=0.0` цифры образуют компактные островки. При `n_neighbors=50, min_dist=0.5` облака становятся шире. Эти параметры не означают «правильное число кластеров» и не выбираются только по красоте: настройки сравнивают по устойчивости, метрике trustworthiness и цели анализа.

15. **Почему первое измерение времени UMAP может быть завышено?**

   **Простыми словами.** Библиотека UMAP использует Numba, которая переводит часть Python-кода в быстрый машинный код во время первого запуска. Компиляция занимает время, поэтому первый вызов включает и подготовку, и само вычисление.

   **Как это работает.** Такой запуск называют «холодным». При следующих вызовах скомпилированные функции уже находятся в памяти или кэше, и измеряется в основном работа алгоритма. На время также влияют загрузка процессора, число потоков, размер данных, параметры и версии библиотек. Поэтому одно число без описания окружения мало что доказывает.

   **Пример и вывод.** В работе первая конфигурация UMAP выполнялась заметно дольше двух последующих. Для честного сравнения полезно отдельно показать холодный запуск, выполнить прогрев, повторить измерение несколько раз и использовать медиану. Нельзя делать общий вывод «UMAP всегда медленнее t-SNE» по одному небольшому набору.

16. **Почему двумерная визуализация не доказывает существование истинных кластеров?**

   **Простыми словами.** При переходе из 64 измерений в 2 часть информации неизбежно теряется. Алгоритм вынужден раздвигать одни точки и сближать другие. Поэтому на плоскости могут появиться убедительные островки даже тогда, когда в исходных данных нет чётких естественных границ.

   **Как это работает.** t-SNE и UMAP специально оптимизируют вид соседств, а выбранные параметры влияют на плотность и разделение групп. Цветные метки также усиливают впечатление упорядоченности. Настоящий кластер должен подтверждаться не только картинкой, но и устойчивостью к параметрам, метриками качества, результатами других методов и предметным смыслом признаков.

   **Пример и вывод.** Для `load_digits` известны правильные классы цифр, поэтому цвет помогает оценить, смешиваются ли похожие изображения. В реальной задаче меток может не быть. Тогда 2D-проекцию используют как источник гипотез: находят кандидатов на группы, а затем проверяют их в исходном пространстве и на новых данных.
